# validations_silver.py

## Purpose
Runs a suite of data quality validations on all Silver layer tables. This is a read-only notebook: it does not modify data, only reports.

## Strategy
- Separate notebook from the transformation pipelines
- Read-only: does not write or modify tables
- Idempotent: can be run as many times as needed
- Each check reports PASS/FAIL with details of problematic rows
- Final summary with total checks, passed, and failed

## Implemented checks

### 1. Primary key uniqueness (SCD1)
- `dim_edition`: `edition_id` unique
- `dim_gender`: `gender_id` unique
- `dim_position`: `position_id` unique
- `dim_nationality`: `nationality_id` unique

### 2. Current record uniqueness (SCD2)
- `dim_league`: one `Is_current = true` per `league_code`
- `dim_club`: one `Is_current = true` per `(club_code, gender_id, league_id)`
- `dim_playstyle`: one `Is_current = true` per `(playstyle_code, is_plus)`
- `dim_player`: one `Is_current = true` per `player_id`

### 3. Referential integrity
- `bridge_player_position` → `dim_player` and `dim_position`
- `bridge_player_playstyle` → `dim_player` and `dim_playstyle`
- `fact_player_snapshot` → `dim_player` and `dim_edition`
- `fact_player_facets` → `dim_player`

### 4. Valid metric ranges
- Card facets (`overall_rating`, `pace`, `shooting`, etc.): between 1 and 99
- Detailed attributes: between 1 and 99

### 5. Completeness of critical metrics
- `fact_player_snapshot`: no NULLs in `overall_rating`, `pace`, `shooting`, etc.

### 6. Unique grain in fact tables
- `fact_player_snapshot`: unique by `(player_snapshot_key)`
- `fact_player_facets`: unique by `(player_snapshot_key, faceta)`

### 7. SCD2 consistency
- `End_date > Start_date` when `End_date` is not NULL
- One `Is_current = true` per business key

## Output
- Results printed to console (PASS/FAIL per check)
- Final summary with total checks executed
- Details of problematic rows (limited to 10 per check)

## Notes
- If a check fails, a sample of problematic rows is displayed
- The notebook runs at the end of the pipeline, after all transformations

FUNCIONES HELPER

In [0]:
from pyspark.sql.functions import col

# ===== Global results log =====
results = []

def log_check(check_name, table, passed, details_df=None):
    status = "PASS" if passed else "FAIL"
    print(f"[{status}] {check_name} — {table}")
    results.append({"check": check_name, "table": table, "status": status})
    if not passed and details_df is not None:
        display(details_df.limit(10))


def check_pk_uniqueness(table, pk_columns, filter_current=False):
    """Check that the primary key is unique."""
    cols = ", ".join(pk_columns)
    where = "WHERE Is_current = true" if filter_current else ""

    dup = spark.sql(f"""
        SELECT {cols}, COUNT(*) AS cnt
        FROM {table}
        {where}
        GROUP BY {cols}
        HAVING COUNT(*) > 1
    """)

    check_name = f"PK uniqueness ({cols})"
    log_check(check_name, table, dup.count() == 0, dup)


def check_fk_integrity(source_table, fk_column, target_table, target_pk, filter_target_current=False):
    """Check referential integrity."""
    target_where = "AND t.Is_current = true" if filter_target_current else ""

    orphans = spark.sql(f"""
        SELECT s.{fk_column}
        FROM {source_table} s
        LEFT JOIN {target_table} t 
            ON s.{fk_column} = t.{target_pk} {target_where}
        WHERE t.{target_pk} IS NULL
    """)

    check_name = f"FK integrity ({fk_column})"
    log_check(check_name, source_table, orphans.count() == 0, orphans)


def check_range(table, column, min_val, max_val):
    """Check that a column is within a range."""
    invalid = spark.sql(f"""
        SELECT * FROM {table}
        WHERE {column} < {min_val} OR {column} > {max_val}
    """)

    check_name = f"Range {column} [{min_val}-{max_val}]"
    log_check(check_name, table, invalid.count() == 0, invalid)


def check_no_nulls(table, columns):
    """Check that there are no NULLs in the given columns."""
    where = " OR ".join([f"{c} IS NULL" for c in columns])

    nulls = spark.sql(f"""
        SELECT * FROM {table}
        WHERE {where}
    """)

    check_name = f"No nulls ({', '.join(columns)})"
    log_check(check_name, table, nulls.count() == 0, nulls)


def check_grain(table, grain_columns):
    """Check that the grain is unique."""
    cols = ", ".join(grain_columns)

    dup = spark.sql(f"""
        SELECT {cols}, COUNT(*) AS cnt
        FROM {table}
        GROUP BY {cols}
        HAVING COUNT(*) > 1
    """)

    check_name = f"Unique grain ({cols})"
    log_check(check_name, table, dup.count() == 0, dup)


def check_scd2_consistency(table, business_key):
    """Check SCD2 consistency: End_date > Start_date and a single Is_current."""
    # End_date > Start_date
    invalid = spark.sql(f"""
        SELECT {business_key}, Start_date, End_date
        FROM {table}
        WHERE End_date IS NOT NULL AND End_date <= Start_date
    """)
    log_check("SCD2 date consistency", table, invalid.count() == 0, invalid)

    # Only one Is_current = true per business key
    dup = spark.sql(f"""
        SELECT {business_key}, COUNT(*) AS cnt
        FROM {table}
        WHERE Is_current = true
        GROUP BY {business_key}
        HAVING COUNT(*) > 1
    """)
    log_check("SCD2 single current", table, dup.count() == 0, dup)

In [0]:
# --- Validation configuration per table ---
VALIDATIONS = {
    "fifa.silver.dim_edition": {
        "pk": ["edition_id"],
        "current": False,
    },
    "fifa.silver.dim_gender": {
        "pk": ["gender_id"],
        "current": False,
    },
    "fifa.silver.dim_position": {
        "pk": ["position_id"],
        "current": False,
    },
    "fifa.silver.dim_nationality": {
        "pk": ["nationality_id"],
        "current": False,
    },
    "fifa.silver.dim_league": {
        "pk": ["league_code"],
        "current": True,
        "scd2_key": "league_code",
    },
    "fifa.silver.dim_club": {
        "pk": ["club_code", "gender_id"],
        "current": True,
        "scd2_key": "club_code",
    },
    "fifa.silver.dim_playstyle": {
        "pk": ["playstyle_code", "is_plus"],
        "current": True,
        "scd2_key": "playstyle_code",
    },
    "fifa.silver.dim_player": {
        "pk": ["player_id"],
        "current": True,
        "scd2_key": "player_id",
    },
}

In [0]:
# --- Validations for dim_edition ---
check_pk_uniqueness("fifa.silver.dim_edition", ["edition_id"])
check_no_nulls("fifa.silver.dim_edition", ["edition_id", "edition_name"])

In [0]:
# --- Validations for dim_gender ---
check_pk_uniqueness("fifa.silver.dim_gender", ["gender_id"])
check_no_nulls("fifa.silver.dim_gender", ["gender_id", "gender_name"])

In [0]:
# --- Validations for dim_position ---
check_pk_uniqueness("fifa.silver.dim_position", ["position_id"])
check_no_nulls("fifa.silver.dim_position", ["position_id", "position_name"])

In [0]:
# --- Validations for dim_nationality ---
check_pk_uniqueness("fifa.silver.dim_nationality", ["nationality_id"])
check_no_nulls("fifa.silver.dim_nationality", ["nationality_id", "nationality_name"])

In [0]:
# --- Validations for dim_league ---
check_pk_uniqueness("fifa.silver.dim_league", ["league_code"], filter_current=True)
check_no_nulls("fifa.silver.dim_league", ["league_code", "league_name"])
check_scd2_consistency("fifa.silver.dim_league", "league_code")

In [0]:
# --- Validations for dim_club ---
check_pk_uniqueness("fifa.silver.dim_club", ["club_code", "gender_id", "league_id"], filter_current=True)
check_no_nulls("fifa.silver.dim_club", ["club_code", "club_name", "gender_id"])
check_fk_integrity("fifa.silver.dim_club", "gender_id", "fifa.silver.dim_gender", "gender_id")
check_fk_integrity("fifa.silver.dim_club", "league_id", "fifa.silver.dim_league", "league_id", filter_target_current=True)
check_scd2_consistency("fifa.silver.dim_club", "club_code, gender_id, league_id")

In [0]:
# --- Validations for dim_playstyle ---
check_pk_uniqueness("fifa.silver.dim_playstyle", ["playstyle_code", "is_plus"], filter_current=True)
check_no_nulls("fifa.silver.dim_playstyle", ["playstyle_code", "playstyle_name", "is_plus"])
check_scd2_consistency("fifa.silver.dim_playstyle", "playstyle_code, is_plus")

In [0]:
# --- Validations for dim_player ---
check_pk_uniqueness("fifa.silver.dim_player", ["player_id"], filter_current=True)
check_no_nulls("fifa.silver.dim_player", ["player_id", "first_name", "last_name"])
check_scd2_consistency("fifa.silver.dim_player", "player_id")

In [0]:
# --- Validations for bridge_player_position ---
check_grain("fifa.silver.bridge_player_position", ["player_snapshot_key", "position_id", "position_type"])
check_fk_integrity("fifa.silver.bridge_player_position", "player_snapshot_key", "fifa.silver.dim_player", "player_snapshot_key")
check_fk_integrity("fifa.silver.bridge_player_position", "position_id", "fifa.silver.dim_position", "position_id")
check_no_nulls("fifa.silver.bridge_player_position", ["player_snapshot_key", "player_id", "position_id", "position_type"])


In [0]:
# --- Validations for bridge_player_playstyle ---
check_grain("fifa.silver.bridge_player_playstyle", ["player_snapshot_key", "playstyle_id", "is_plus"])
check_fk_integrity("fifa.silver.bridge_player_playstyle", "player_snapshot_key", "fifa.silver.dim_player", "player_snapshot_key")
check_fk_integrity("fifa.silver.bridge_player_playstyle", "playstyle_id", "fifa.silver.dim_playstyle", "playstyle_id")
check_no_nulls("fifa.silver.bridge_player_playstyle", ["player_snapshot_key", "player_id", "playstyle_id", "is_plus"])

In [0]:
# --- Validations for fact_player_snapshot ---
check_grain("fifa.silver.fact_player_snapshot", ["player_id", "snapshot_date"])
check_fk_integrity("fifa.silver.fact_player_facets", "player_snapshot_key", "fifa.silver.dim_player", "player_snapshot_key")
check_fk_integrity("fifa.silver.fact_player_snapshot", "edition_id", "fifa.silver.dim_edition", "edition_id")
check_no_nulls("fifa.silver.fact_player_snapshot", ["player_id", "snapshot_date", "overall_rating"])

# Range 1-99 in facetas
for metric in ["overall_rating", "pace", "shooting", "passing", "dribbling", "defending", "physicality"]:
    check_range("fifa.silver.fact_player_snapshot", metric, 1, 99)

In [0]:
# --- Validations for fact_player_facets ---
check_grain("fifa.silver.fact_player_facets", ["player_id", "snapshot_date", "faceta"])
check_fk_integrity("fifa.silver.fact_player_facets", "player_id", "fifa.silver.dim_player", "player_id")
check_no_nulls("fifa.silver.fact_player_facets", ["player_id", "snapshot_date", "faceta", "valor"])
check_range("fifa.silver.fact_player_facets", "valor", 1, 99)


In [0]:
import pandas as pd

summary = pd.DataFrame(results)
summary["passed"] = summary["status"] == "PASS"

total = len(summary)
passed = summary["passed"].sum()
failed = total - passed

print(f"\n{'='*60}")
print(f"RESUMEN DE VALIDACIONES")
print(f"{'='*60}")
print(f"Total checks:  {total}")
print(f"PASS:          {passed}")
print(f"FAIL:          {failed}")
print(f"{'='*60}\n")

if failed > 0:
    print("Checks fallidos:")
    display(summary[~summary["passed"]])
else:
    print("✅ Todos los checks pasaron")

In [0]:
# para ver el total de filas de cada tabla
tables = [
    "fifa.silver.dim_edition",
    "fifa.silver.dim_gender",
    "fifa.silver.dim_position",
    "fifa.silver.dim_nationality",
    "fifa.silver.dim_league",
    "fifa.silver.dim_club",
    "fifa.silver.dim_playstyle",
    "fifa.silver.dim_player",
    "fifa.silver.bridge_player_position",
    "fifa.silver.bridge_player_playstyle",
    "fifa.silver.fact_player_snapshot",
    "fifa.silver.fact_player_facets"
]

for t in tables:
    count = spark.read.table(t).count()
    print(f"{t}: {count} filas")